In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Standard Library Imports

# ─────────────────────────────────────────────────────────────────────────────
# Third-Party Imports
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import List

# Define Seaborn style
def set_academic_style():
    
    # Color Universal Design Palette
    color_palette = [
        "#377eb8",  # azul
        "#ff7f00",  # naranja
        "#4daf4a",  # verde
        "#f781bf",  # rosa
        "#a65628",  # marrón
        "#984ea3",  # púrpura
        "#999999",  # gris
        "#e41a1c",  # rojo
        "#dede00"   # amarillo
    ]

    # Style
    sns.set(
        style="whitegrid",
        palette=color_palette,
        context="paper",
        font_scale=1.5
    )

    # Additional Matplotlib Config
    plt.rcParams.update({
        "figure.figsize": (10, 6),
        "axes.titlesize": 18,
        "axes.labelsize": 14,
        "axes.labelweight": "bold",
        "axes.grid": True,
        "grid.alpha": 0.6,
        "grid.linestyle": "--",
        "legend.fontsize": 12,
        "legend.title_fontsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "savefig.dpi": 300,
        "savefig.format": "pdf"
    })

set_academic_style()


# Input

In [4]:
def datasetsCreator(csv_files):
    # Leer y procesar cada archivo
    datasets: List[pl.DataFrame] = []
    for filepath in csv_files:
        df: pl.DataFrame = pl.read_csv(filepath)
        df = df.with_columns([
            (
                (pl.col("timestamp") - pl.col("timestamp").min())
                .cast(pl.Duration("ns")) + pl.lit(0).cast(pl.Datetime("ns"))
            ).alias("timestamp"),
            pl.col("value").cast(pl.Int64)
        ])
        # Filter PTP metrics
        df_ptp = df.filter(
            pl.col("metric").str.contains("PPS") | 
            pl.col("metric").str.contains("offset")
            )

        # PTP Correspondence
        ptp_map = {
            "PPS_3_PPSAnalyzer": "TimeStick_hw_offset",
            "PPS_4_PPSAnalyzer": "Intel710_hw_offset",
            "PPS_5_PPSAnalyzer": "Intel810_hw_offset",
            "offset_TimeStick_PTP_Log": "TimeStick_sw_offset",
            "offset_Intel710_PTP_Log": "Intel710_sw_offset",
            "offset_Intel810_PTP_Log": "Intel810_sw_offset",
        }

        # Name Replacement
        df_ptp = df_ptp.with_columns(
            pl.col("metric").replace(ptp_map).alias("PTP_metric"),
            pl.col("value").alias("offset_ns")
        ).drop("metric", "value")

        # Hardware PTP Dataframe
        df_hw_ptp = df_ptp.filter(
            pl.col("PTP_metric").str.contains("hw")
        ).rename({
            "offset_ns": "offset_hw_ns",
            "PTP_metric":"PTP_client",
        }).with_columns(
            pl.col("PTP_client").replace(
                {
                    "TimeStick_hw_offset": "TimeStick",
                    "Intel710_hw_offset": "Intel710",
                    "Intel810_hw_offset": "Intel810",
                }
            )
        )
        
        datasets.append(df_hw_ptp)
    return datasets

def load_datasets(base_dir: str, prefix: str, durations: List[int], suffix: str) -> List[List[pl.DataFrame]]:
    all_datasets: List[List[pl.DataFrame]] = []
    for duration in durations:
        time_label = f"T{duration}"
        dir_path = Path(f"{base_dir}/{prefix}-{time_label}-{suffix}")
        pattern = f"{prefix}-{time_label}-{suffix}_*.csv"
        csv_files: List[Path] = list(dir_path.glob(pattern))
        datasets: List[pl.DataFrame] = datasetsCreator(csv_files)
        all_datasets.append(datasets)
    return all_datasets

In [11]:
# Calibration
dataset_calibration_IDLE = load_datasets("../output/Ly3_C_IDLE", "Ly3_CTOp_L0", [120], "IDLE")[0][1]

In [12]:

def compute_stats(df: pl.DataFrame) -> pl.DataFrame:
    """
    Calcula estadísticas descriptivas por cliente (PTP_client).
    """
    stats: pl.DataFrame = (
        df.group_by("PTP_client")
          .agg([
              pl.len().alias("count"),
              pl.col("offset_hw_ns").mean().alias("mean"),
              pl.col("offset_hw_ns").std().alias("std"),
              pl.col("offset_hw_ns").min().alias("min"),
              pl.col("offset_hw_ns").quantile(0.25).alias("p25"),
              pl.col("offset_hw_ns").median().alias("median"),
              pl.col("offset_hw_ns").quantile(0.75).alias("p75"),
              pl.col("offset_hw_ns").max().alias("max"),
              pl.col("offset_hw_ns").var().alias("variance"),
          ])
          .sort("PTP_client")
    )
    return stats

stats_df: pl.DataFrame = compute_stats(dataset_calibration_IDLE)
print(stats_df)


shape: (3, 10)
┌────────────┬───────┬────────────┬───────────┬───┬────────┬───────┬─────┬────────────┐
│ PTP_client ┆ count ┆ mean       ┆ std       ┆ … ┆ median ┆ p75   ┆ max ┆ variance   │
│ ---        ┆ ---   ┆ ---        ┆ ---       ┆   ┆ ---    ┆ ---   ┆ --- ┆ ---        │
│ str        ┆ u32   ┆ f64        ┆ f64       ┆   ┆ f64    ┆ f64   ┆ i64 ┆ f64        │
╞════════════╪═══════╪════════════╪═══════════╪═══╪════════╪═══════╪═════╪════════════╡
│ Intel710   ┆ 7200  ┆ 325.608194 ┆ 23.081241 ┆ … ┆ 325.0  ┆ 341.0 ┆ 409 ┆ 532.743675 │
│ Intel810   ┆ 7200  ┆ -36.896111 ┆ 8.525265  ┆ … ┆ -37.0  ┆ -31.0 ┆ -10 ┆ 72.680135  │
│ TimeStick  ┆ 7200  ┆ 163.650972 ┆ 28.949491 ┆ … ┆ 162.0  ┆ 175.0 ┆ 864 ┆ 838.073051 │
└────────────┴───────┴────────────┴───────────┴───┴────────┴───────┴─────┴────────────┘
